<a href="https://colab.research.google.com/github/mostafadentist/healthcare-data-analytics/blob/main/dental_materials_company_data_analytics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from datetime import datetime, timedelta
import random
# Create subplot figure
from plotly.subplots import make_subplots
# Set random seed for reproducibility
np.random.seed(42)
random.seed(42)

# Generate synthetic product sales data
def generate_product_sales_data():
    # Product categories and their details
    products = {
        'Surgical Kits': {
            'items': ['Basic Surgical Kit', 'Advanced Implant Kit', 'Oral Surgery Set', 'Wisdom Tooth Extraction Kit'],
            'price_range': (250, 1500),
            'volume_range': (50, 200)
        },
        'Dental Composites': {
            'items': ['Universal Composite A2', 'Universal Composite A3', 'Flowable Composite', 'Bulk Fill Composite'],
            'price_range': (45, 180),
            'volume_range': (200, 800)
        },
        'Impression Materials': {
            'items': ['Alginate Powder', 'Silicone Light Body', 'Silicone Heavy Body', 'Polyether Material'],
            'price_range': (30, 250),
            'volume_range': (150, 600)
        },
        'Hand Instruments': {
            'items': ['Explorer Set', 'Scaler Kit', 'Extraction Forceps', 'Elevator Set', 'Mirror & Probe Set'],
            'price_range': (15, 300),
            'volume_range': (100, 500)
        },
        'Dental Implants': {
            'items': ['Titanium Implant 3.5mm', 'Titanium Implant 4.0mm', 'Zirconia Implant', 'Mini Implant'],
            'price_range': (150, 800),
            'volume_range': (30, 150)
        },
        'Bleaching Products': {
            'items': ['In-Office Whitening Kit', 'Take-Home Kit 16%', 'Take-Home Kit 22%', 'LED Whitening System'],
            'price_range': (25, 400),
            'volume_range': (100, 400)
        }
    }

    # Customer segments
    customer_types = ['Private Clinics', 'Hospital Chains', 'Dental Schools', 'Group Practices', 'Solo Practitioners']
    regions = ['North America', 'Europe', 'Asia Pacific', 'Latin America', 'Middle East']

    data = []

    for month in pd.date_range('2023-01', '2024-12', freq='MS'):
        for category, details in products.items():
            for item in details['items']:
                for region in regions:
                    for customer_type in customer_types:
                        # Seasonal factors
                        seasonal_factor = 1 + 0.2 * np.sin((month.month - 3) * np.pi / 6)

                        # Regional multipliers
                        regional_multiplier = {'North America': 1.3, 'Europe': 1.2, 'Asia Pacific': 1.4,
                                              'Latin America': 0.9, 'Middle East': 1.0}[region]

                        base_volume = np.random.randint(*details['volume_range'])
                        volume = int(base_volume * seasonal_factor * regional_multiplier * np.random.uniform(0.5, 1.5))
                        unit_price = np.random.uniform(*details['price_range'])

                        data.append({
                            'Date': month,
                            'Category': category,
                            'Product': item,
                            'Region': region,
                            'Customer_Type': customer_type,
                            'Units_Sold': volume,
                            'Unit_Price': unit_price,
                            'Revenue': volume * unit_price,
                            'Cost': volume * unit_price * np.random.uniform(0.4, 0.6),
                            'Discount_Applied': np.random.uniform(0, 0.15)
                        })

    return pd.DataFrame(data)

# Generate data
df_sales = generate_product_sales_data()
df_sales['Profit'] = df_sales['Revenue'] - df_sales['Cost']
df_sales['Profit_Margin'] = (df_sales['Profit'] / df_sales['Revenue'] * 100).round(1)

# Create comprehensive sales dashboard
fig = make_subplots(
    rows=3, cols=2,
    subplot_titles=('Revenue by Product Category', 'Monthly Sales Trend',
                    'Regional Performance', 'Top Selling Products',
                    'Customer Segment Analysis', 'Profit Margin by Category'),
    specs=[[{'type': 'bar'}, {'type': 'scatter'}],
           [{'type': 'geo'}, {'type': 'bar'}],
           [{'type': 'sunburst'}, {'type': 'box'}]]
)

# 1. Revenue by category
category_revenue = df_sales.groupby('Category')['Revenue'].sum().sort_values(ascending=False)
fig.add_trace(
    go.Bar(x=category_revenue.index, y=category_revenue.values,
           marker_color=['#FF6B6B', '#4ECDC4', '#45B7D1', '#96E6B3', '#F7DC6F', '#BB8FCE'],
           text=[f'${v/1e6:.1f}M' for v in category_revenue.values],
           textposition='auto'),
    row=1, col=1
)

# 2. Monthly trend
monthly_sales = df_sales.groupby(['Date', 'Category'])['Revenue'].sum().reset_index()
for category in df_sales['Category'].unique():
    cat_data = monthly_sales[monthly_sales['Category'] == category]
    fig.add_trace(
        go.Scatter(x=cat_data['Date'], y=cat_data['Revenue'],
                   name=category, mode='lines', line=dict(width=2)),
        row=1, col=2
    )

# 3. Top selling products
top_products = df_sales.groupby('Product')['Units_Sold'].sum().sort_values(ascending=False).head(10)
fig.add_trace(
    go.Bar(x=top_products.values, y=top_products.index,
           orientation='h', marker_color='teal',
           text=top_products.values, textposition='auto'),
    row=2, col=2
)

# 4. Customer segment sunburst
segment_data = df_sales.groupby(['Customer_Type', 'Category'])['Revenue'].sum().reset_index()
fig.add_trace(
    go.Sunburst(
        labels=segment_data['Customer_Type'].tolist() + segment_data['Category'].tolist(),
        parents=[''] * len(segment_data['Customer_Type'].unique()) * len(segment_data['Category'].unique()) +
                segment_data['Customer_Type'].tolist(),
        values=segment_data['Revenue'].tolist() + segment_data['Revenue'].tolist()
    ),
    row=3, col=1
)

# 5. Profit margin distribution
fig.add_trace(
    go.Box(x=df_sales['Category'], y=df_sales['Profit_Margin'],
           marker_color='orange'),
    row=3, col=2
)

fig.update_layout(height=1000, showlegend=True, title_text="Dental Supplies Sales Performance Dashboard (2023-2024)")
fig.show()

print(f"Total Revenue: ${df_sales['Revenue'].sum():,.2f}")
print(f"Average Profit Margin: {df_sales['Profit_Margin'].mean():.1f}%")
print(f"Total Units Sold: {df_sales['Units_Sold'].sum():,}")

Total Revenue: $1,033,740,617.88
Average Profit Margin: 50.1%
Total Units Sold: 4,738,859


In [4]:
# Generate inventory data
def generate_inventory_data():
    products = {
        'Surgical Kits': {'sku_prefix': 'SK', 'lead_time': 14, 'shelf_life': 1095},
        'Dental Composites': {'sku_prefix': 'DC', 'lead_time': 7, 'shelf_life': 730},
        'Impression Materials': {'sku_prefix': 'IM', 'lead_time': 10, 'shelf_life': 365},
        'Hand Instruments': {'sku_prefix': 'HI', 'lead_time': 21, 'shelf_life': 3650},
        'Dental Implants': {'sku_prefix': 'DI', 'lead_time': 30, 'shelf_life': 1825},
        'Bleaching Products': {'sku_prefix': 'BP', 'lead_time': 7, 'shelf_life': 365}
    }

    warehouses = ['New York', 'Los Angeles', 'Chicago', 'Houston', 'Amsterdam', 'Singapore']

    data = []

    for month in pd.date_range('2024-01', '2024-12', freq='MS'):
        for category, details in products.items():
            for i in range(5):  # 5 SKUs per category
                sku = f"{details['sku_prefix']}-{1000 + i}"

                for warehouse in warehouses:
                    # Calculate inventory metrics
                    opening_stock = np.random.randint(100, 1000)
                    received = np.random.randint(0, 500)
                    sold = np.random.randint(50, 400)
                    closing_stock = opening_stock + received - sold

                    # Safety stock and reorder points
                    daily_usage = sold / 30
                    safety_stock = daily_usage * details['lead_time'] * 1.5
                    reorder_point = safety_stock + (daily_usage * details['lead_time'])

                    # Expiry tracking
                    days_to_expiry = np.random.randint(30, details['shelf_life'])

                    data.append({
                        'Month': month,
                        'Category': category,
                        'SKU': sku,
                        'Warehouse': warehouse,
                        'Opening_Stock': opening_stock,
                        'Received': received,
                        'Sold': sold,
                        'Closing_Stock': max(0, closing_stock),
                        'Safety_Stock': int(safety_stock),
                        'Reorder_Point': int(reorder_point),
                        'Lead_Time_Days': details['lead_time'],
                        'Days_to_Expiry': days_to_expiry,
                        'Stock_Value': closing_stock * np.random.uniform(20, 500),
                        'Turnover_Ratio': (sold * 12) / max(1, (opening_stock + closing_stock) / 2)
                    })

    return pd.DataFrame(data)

# Generate data
df_inventory = generate_inventory_data()

# Calculate additional metrics
df_inventory['Stock_Status'] = df_inventory.apply(
    lambda x: 'Critical' if x['Closing_Stock'] < x['Safety_Stock']
    else 'Low' if x['Closing_Stock'] < x['Reorder_Point']
    else 'Optimal' if x['Closing_Stock'] < x['Reorder_Point'] * 2
    else 'Excess', axis=1
)

df_inventory['Expiry_Risk'] = df_inventory['Days_to_Expiry'].apply(
    lambda x: 'High' if x < 90 else 'Medium' if x < 180 else 'Low'
)

# Create inventory dashboard
fig = make_subplots(
    rows=3, cols=2,
    subplot_titles=('Inventory Turnover by Category', 'Stock Levels by Warehouse',
                    'Stock Status Distribution', 'Expiry Risk Analysis',
                    'Monthly Stock Movement', 'Warehouse Efficiency'),
    specs=[[{'type': 'bar'}, {'type': 'bar'}],
           [{'type': 'pie'}, {'type': 'sunburst'}],
           [{'type': 'scatter'}, {'type': 'heatmap'}]]
)

# 1. Turnover by category
turnover_by_cat = df_inventory.groupby('Category')['Turnover_Ratio'].mean().sort_values(ascending=False)
fig.add_trace(
    go.Bar(x=turnover_by_cat.index, y=turnover_by_cat.values,
           marker_color='green', text=turnover_by_cat.round(1).values,
           textposition='auto'),
    row=1, col=1
)

# 2. Stock levels by warehouse
warehouse_stock = df_inventory.groupby('Warehouse')['Stock_Value'].sum().sort_values(ascending=False)
fig.add_trace(
    go.Bar(x=warehouse_stock.index, y=warehouse_stock.values,
           marker_color='blue', text=[f'${v/1e6:.1f}M' for v in warehouse_stock.values],
           textposition='auto'),
    row=1, col=2
)

# 3. Stock status pie
status_dist = df_inventory['Stock_Status'].value_counts()
fig.add_trace(
    go.Pie(labels=status_dist.index, values=status_dist.values,
           marker=dict(colors=['#FF4444', '#FFA500', '#00CC00', '#FFD700']),
           hole=0.3),
    row=2, col=1
)

# 4. Expiry risk sunburst
expiry_data = df_inventory.groupby(['Expiry_Risk', 'Category']).size().reset_index(name='Count')
fig.add_trace(
    go.Sunburst(
        labels=expiry_data['Expiry_Risk'].tolist() + expiry_data['Category'].tolist(),
        parents=[''] * len(expiry_data['Expiry_Risk'].unique()) * len(expiry_data['Category'].unique()) +
                expiry_data['Expiry_Risk'].tolist(),
        values=expiry_data['Count'].tolist() + expiry_data['Count'].tolist(),
        marker=dict(colors=['red', 'orange', 'green'])
    ),
    row=2, col=2
)

# 5. Monthly stock movement
monthly_movement = df_inventory.groupby('Month').agg({
    'Received': 'sum',
    'Sold': 'sum',
    'Closing_Stock': 'mean'
}).reset_index()

fig.add_trace(
    go.Scatter(x=monthly_movement['Month'], y=monthly_movement['Received'],
               name='Received', mode='lines+markers', line=dict(color='blue', width=2)),
    row=3, col=1
)
fig.add_trace(
    go.Scatter(x=monthly_movement['Month'], y=monthly_movement['Sold'],
               name='Sold', mode='lines+markers', line=dict(color='green', width=2)),
    row=3, col=1
)

# 6. Warehouse efficiency heatmap
warehouse_eff = df_inventory.pivot_table(values='Turnover_Ratio',
                                         index='Warehouse',
                                         columns='Category',
                                         aggfunc='mean')
fig.add_trace(
    go.Heatmap(z=warehouse_eff.values, x=warehouse_eff.columns, y=warehouse_eff.index,
               colorscale='RdYlGn', text=warehouse_eff.round(1).values,
               texttemplate='%{text}', textfont={"size": 10}),
    row=3, col=2
)

fig.update_layout(height=1000, showlegend=True, title_text="Inventory Management Dashboard")
fig.show()

print(f"Total Inventory Value: ${df_inventory['Stock_Value'].sum():,.2f}")
print(f"Average Turnover Ratio: {df_inventory['Turnover_Ratio'].mean():.2f}")
print(f"Critical Stock Items: {(df_inventory['Stock_Status'] == 'Critical').sum()}")
print(f"High Expiry Risk Items: {(df_inventory['Expiry_Risk'] == 'High').sum()}")

Total Inventory Value: $329,074,729.36
Average Turnover Ratio: 28.05
Critical Stock Items: 276
High Expiry Risk Items: 185


In [5]:
# Generate customer order data
def generate_customer_data():
    # Customer database
    np.random.seed(42)
    customers = []
    customer_types = {
        'Private Clinic': {'count': 150, 'order_freq': 'high', 'avg_order': (500, 5000)},
        'Hospital Chain': {'count': 30, 'order_freq': 'very_high', 'avg_order': (5000, 50000)},
        'Dental School': {'count': 20, 'order_freq': 'medium', 'avg_order': (2000, 20000)},
        'Group Practice': {'count': 80, 'order_freq': 'high', 'avg_order': (1000, 10000)},
        'Solo Practitioner': {'count': 200, 'order_freq': 'low', 'avg_order': (100, 2000)}
    }

    # Generate customer list
    customer_id = 1000
    for ctype, details in customer_types.items():
        for i in range(details['count']):
            customers.append({
                'Customer_ID': f'C{customer_id}',
                'Customer_Name': f'{ctype} #{i+1}',
                'Type': ctype,
                'Order_Frequency': details['order_freq'],
                'Avg_Order_Range': details['avg_order'],
                'Registration_Date': pd.Timestamp('2020-01-01') + pd.Timedelta(days=np.random.randint(0, 1460))
            })
            customer_id += 1

    customers_df = pd.DataFrame(customers)

    # Generate order history
    orders = []
    order_id = 10000

    freq_map = {'very_high': 8, 'high': 4, 'medium': 2, 'low': 1}

    for _, customer in customers_df.iterrows():
        orders_per_month = freq_map[customer['Order_Frequency']]

        for month in pd.date_range('2023-01', '2024-12', freq='MS'):
            for _ in range(np.random.poisson(orders_per_month)):
                order_value = np.random.uniform(*customer['Avg_Order_Range'])

                # Seasonal adjustments
                if month.month in [11, 12]:  # Year-end bulk orders
                    order_value *= 1.3

                # Customer loyalty discount
                months_active = (month - customer['Registration_Date']).days / 30
                loyalty_discount = min(0.15, months_active / 1000)  # Max 15% discount

                orders.append({
                    'Order_ID': f'ORD{order_id}',
                    'Customer_ID': customer['Customer_ID'],
                    'Customer_Type': customer['Type'],
                    'Order_Date': month + pd.Timedelta(days=np.random.randint(0, 28)),
                    'Order_Value': order_value,
                    'Discount_Applied': loyalty_discount,
                    'Net_Value': order_value * (1 - loyalty_discount),
                    'Payment_Terms': np.random.choice(['Net 30', 'Net 60', 'Immediate', 'Net 90']),
                    'Order_Status': np.random.choice(['Delivered', 'Pending', 'Processing'], p=[0.85, 0.10, 0.05]),
                    'Items_Count': np.random.randint(1, 20),
                    'Repeat_Customer': months_active > 180
                })
                order_id += 1

    return pd.DataFrame(orders)

# Generate data
df_orders = generate_customer_data()

# Calculate customer metrics
customer_metrics = df_orders.groupby('Customer_ID').agg({
    'Net_Value': ['sum', 'mean', 'count'],
    'Items_Count': 'sum',
    'Repeat_Customer': 'max'
}).reset_index()
customer_metrics.columns = ['Customer_ID', 'Total_Value', 'Avg_Order_Value', 'Order_Count', 'Total_Items', 'Is_Repeat']

# Customer segmentation
customer_metrics['Segment'] = pd.cut(customer_metrics['Total_Value'],
                                     bins=[0, 10000, 50000, 200000, float('inf')],
                                     labels=['Bronze', 'Silver', 'Gold', 'Platinum'])

# Create customer analytics dashboard
fig = make_subplots(
    rows=3, cols=2,
    subplot_titles=('Customer Segmentation', 'Order Value Distribution',
                    'Customer Type Performance', 'Payment Terms Analysis',
                    'Monthly Order Trends', 'Customer Retention Analysis'),
    specs=[[{'type': 'pie'}, {'type': 'histogram'}],
           [{'type': 'bar'}, {'type': 'pie'}],
           [{'type': 'scatter'}, {'type': 'funnel'}]]
)

# 1. Customer segmentation
segment_counts = customer_metrics['Segment'].value_counts()
fig.add_trace(
    go.Pie(labels=segment_counts.index, values=segment_counts.values,
           marker=dict(colors=['#CD7F32', '#C0C0C0', '#FFD700', '#E5E4E2']),
           hole=0.4),
    row=1, col=1
)

# 2. Order value distribution
fig.add_trace(
    go.Histogram(x=df_orders['Net_Value'], nbinsx=50,
                 marker_color='teal', name='Order Values'),
    row=1, col=2
)

# 3. Customer type performance
type_performance = df_orders.groupby('Customer_Type').agg({
    'Net_Value': 'sum',
    'Order_ID': 'count'
}).reset_index()
type_performance.columns = ['Customer_Type', 'Total_Revenue', 'Order_Count']

fig.add_trace(
    go.Bar(x=type_performance['Customer_Type'],
           y=type_performance['Total_Revenue'],
           text=[f'${v/1e6:.1f}M' for v in type_performance['Total_Revenue']],
           textposition='auto',
           marker_color='purple'),
    row=2, col=1
)

# 4. Payment terms
payment_dist = df_orders['Payment_Terms'].value_counts()
fig.add_trace(
    go.Pie(labels=payment_dist.index, values=payment_dist.values,
           marker=dict(colors=['#FF6B6B', '#4ECDC4', '#45B7D1', '#96E6B3'])),
    row=2, col=2
)

# 5. Monthly order trends
monthly_orders = df_orders.groupby(pd.Grouper(key='Order_Date', freq='M')).agg({
    'Net_Value': 'sum',
    'Order_ID': 'count'
}).reset_index()

fig.add_trace(
    go.Scatter(x=monthly_orders['Order_Date'], y=monthly_orders['Net_Value'],
               mode='lines+markers', line=dict(color='green', width=2),
               name='Revenue'),
    row=3, col=1
)

# 6. Customer retention funnel
total_customers = len(customer_metrics)
active_customers = len(customer_metrics[customer_metrics['Order_Count'] > 1])
regular_customers = len(customer_metrics[customer_metrics['Order_Count'] > 10])
vip_customers = len(customer_metrics[customer_metrics['Segment'].isin(['Gold', 'Platinum'])])

fig.add_trace(
    go.Funnel(
        y=['All Customers', 'Active (>1 order)', 'Regular (>10 orders)', 'VIP (Gold/Platinum)'],
        x=[total_customers, active_customers, regular_customers, vip_customers],
        textinfo='value+percent initial',
        marker=dict(color=['#FF6B6B', '#4ECDC4', '#45B7D1', '#96E6B3'])
    ),
    row=3, col=2
)

fig.update_layout(height=1000, showlegend=False, title_text="Customer Analytics & Order Patterns Dashboard")
fig.show()

print(f"Total Customers: {len(customer_metrics)}")
print(f"Total Orders: {len(df_orders)}")
print(f"Total Revenue: ${df_orders['Net_Value'].sum():,.2f}")
print(f"Average Order Value: ${df_orders['Net_Value'].mean():,.2f}")
print(f"Repeat Customer Rate: {(customer_metrics['Is_Repeat'].sum() / len(customer_metrics) * 100):.1f}%")

/tmp/ipython-input-3352811506.py:136: FutureWarning:

'M' is deprecated and will be removed in a future version, please use 'ME' instead.



Total Customers: 480
Total Orders: 33349
Total Revenue: $262,772,603.77
Average Order Value: $7,879.47
Repeat Customer Rate: 0.0%


In [7]:
# Generate supply chain data
def generate_supply_chain_data():
    suppliers = {
        'Shanghai Medical Supplies': {'location': 'China', 'lead_time': 30, 'reliability': 0.85},
        'German Precision Instruments': {'location': 'Germany', 'lead_time': 14, 'reliability': 0.95},
        'US Dental Manufacturing': {'location': 'USA', 'lead_time': 7, 'reliability': 0.92},
        'Indian Pharma Solutions': {'location': 'India', 'lead_time': 21, 'reliability': 0.80},
        'Japanese Tech Dental': {'location': 'Japan', 'lead_time': 18, 'reliability': 0.93}
    }

    shipping_methods = ['Air Freight', 'Sea Freight', 'Ground Transport', 'Express Courier']

    data = []

    for month in pd.date_range('2023-01', '2024-12', freq='MS'):
        for supplier_name, details in suppliers.items():
            # Generate multiple shipments per month
            shipments = np.random.randint(5, 20)

            for _ in range(shipments):
                base_lead = details['lead_time']
                actual_lead = base_lead + np.random.normal(0, base_lead * 0.1)

                on_time = np.random.random() < details['reliability']

                shipping_method = np.random.choice(shipping_methods)
                if shipping_method == 'Air Freight':
                    cost_multiplier = 3
                    time_reduction = 0.5
                elif shipping_method == 'Express Courier':
                    cost_multiplier = 4
                    time_reduction = 0.3
                elif shipping_method == 'Sea Freight':
                    cost_multiplier = 1
                    time_reduction = 1.5
                else:
                    cost_multiplier = 2
                    time_reduction = 1

                order_value = np.random.uniform(10000, 100000)
                shipping_cost = order_value * 0.05 * cost_multiplier

                data.append({
                    'Month': month,
                    'Supplier': supplier_name,
                    'Country': details['location'],
                    'Shipping_Method': shipping_method,
                    'Order_Value': order_value,
                    'Shipping_Cost': shipping_cost,
                    'Expected_Lead_Time': base_lead,
                    'Actual_Lead_Time': max(1, actual_lead * time_reduction),
                    'On_Time_Delivery': on_time,
                    'Quality_Score': np.random.uniform(85, 100) if on_time else np.random.uniform(70, 85),
                    'Units_Received': np.random.randint(100, 5000),
                    'Defective_Units': np.random.randint(0, 50),
                    'Temperature_Controlled': np.random.choice([True, False]),
                    'Customs_Delay_Days': np.random.randint(0, 5) if details['location'] != 'USA' else 0
                })

    return pd.DataFrame(data)

# Generate logistics data
df_logistics = generate_supply_chain_data()
df_logistics['Lead_Time_Variance'] = df_logistics['Actual_Lead_Time'] - df_logistics['Expected_Lead_Time']
df_logistics['Defect_Rate'] = (df_logistics['Defective_Units'] / df_logistics['Units_Received'] * 100).round(2)

# Create supply chain dashboard
fig = make_subplots(
    rows=3, cols=2,
    subplot_titles=('Supplier Performance Score', 'On-Time Delivery Rate',
                    'Shipping Cost Analysis', 'Lead Time Performance',
                    'Quality Metrics by Supplier', 'Logistics Cost Trend'),
    specs=[[{'type': 'bar'}, {'type': 'indicator'}],
           [{'type': 'box'}, {'type': 'scatter'}],
           [{'type': 'heatmap'}, {'type': 'scatter'}]],
    vertical_spacing=0.12
)

# 1. Supplier performance score
supplier_performance = df_logistics.groupby('Supplier').agg({
    'On_Time_Delivery': lambda x: (x.sum() / len(x) * 100),
    'Quality_Score': 'mean',
    'Lead_Time_Variance': 'mean'
}).reset_index()

supplier_performance['Overall_Score'] = (
    supplier_performance['On_Time_Delivery'] * 0.4 +
    supplier_performance['Quality_Score'] * 0.4 -
    abs(supplier_performance['Lead_Time_Variance']) * 2
)

fig.add_trace(
    go.Bar(x=supplier_performance['Supplier'],
           y=supplier_performance['Overall_Score'],
           marker_color=['green' if s > 85 else 'orange' if s > 75 else 'red'
                        for s in supplier_performance['Overall_Score']],
           text=supplier_performance['Overall_Score'].round(1),
           textposition='auto'),
    row=1, col=1
)

# 2. Overall on-time delivery rate
otd_rate = (df_logistics['On_Time_Delivery'].sum() / len(df_logistics)) * 100
fig.add_trace(
    go.Indicator(
        mode="gauge+number",
        value=otd_rate,
        title={'text': "On-Time Delivery Rate (%)"},
        gauge={'axis': {'range': [0, 100]},
               'bar': {'color': "darkgreen"},
               'steps': [
                   {'range': [0, 60], 'color': "lightgray"},
                   {'range': [60, 80], 'color': "gray"},
                   {'range': [80, 100], 'color': "lightgreen"}],
               'threshold': {'line': {'color': "red", 'width': 4},
                           'thickness': 0.75, 'value': 95}}
    ),
    row=1, col=2
)

# 3. Shipping cost by method
fig.add_trace(
    go.Box(x=df_logistics['Shipping_Method'],
           y=df_logistics['Shipping_Cost'],
           marker_color='blue'),
    row=2, col=1
)

# 4. Lead time performance
monthly_lead = df_logistics.groupby(['Month', 'Supplier'])['Actual_Lead_Time'].mean().reset_index()
for supplier in df_logistics['Supplier'].unique():
    supplier_data = monthly_lead[monthly_lead['Supplier'] == supplier]
    fig.add_trace(
        go.Scatter(x=supplier_data['Month'],
                   y=supplier_data['Actual_Lead_Time'],
                   name=supplier, mode='lines'),
        row=2, col=2
    )

# 5. Quality heatmap
quality_matrix = df_logistics.pivot_table(values='Quality_Score',
                                          index='Supplier',
                                          columns=pd.Grouper(key='Month', freq='Q'),
                                          aggfunc='mean')
fig.add_trace(
    go.Heatmap(z=quality_matrix.values,
               x=[f"Q{i//3+1} {2023 + i//12}" for i in range(quality_matrix.shape[1])],
               y=quality_matrix.index,
               colorscale='RdYlGn',
               text=quality_matrix.round(1).values,
               texttemplate='%{text}',
               textfont={"size": 10}),
    row=3, col=1
)

# 6. Logistics cost trend
monthly_logistics = df_logistics.groupby('Month').agg({
    'Shipping_Cost': 'sum',
    'Order_Value': 'sum'
}).reset_index()
monthly_logistics['Cost_Percentage'] = (monthly_logistics['Shipping_Cost'] /
                                        monthly_logistics['Order_Value'] * 100)

fig.add_trace(
    go.Scatter(x=monthly_logistics['Month'],
               y=monthly_logistics['Cost_Percentage'],
               mode='lines+markers',
               line=dict(color='red', width=2),
               name='Logistics Cost %'),
    row=3, col=2
)

fig.update_layout(height=1000, showlegend=True,
                  title_text="Supply Chain & Logistics Performance Dashboard")
fig.show()

print(f"Average On-Time Delivery Rate: {otd_rate:.1f}%")
print(f"Total Shipping Costs: ${df_logistics['Shipping_Cost'].sum():,.2f}")
print(f"Average Lead Time: {df_logistics['Actual_Lead_Time'].mean():.1f} days")
print(f"Average Quality Score: {df_logistics['Quality_Score'].mean():.1f}")
print(f"Total Defect Rate: {df_logistics['Defect_Rate'].mean():.2f}%")

/tmp/ipython-input-3912104748.py:143: FutureWarning:

'Q' is deprecated and will be removed in a future version, please use 'QE' instead.



Average On-Time Delivery Rate: 88.3%
Total Shipping Costs: $9,369,754.36
Average Lead Time: 15.2 days
Average Quality Score: 90.6
Total Defect Rate: 1.96%
